In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

In [ ]:
DATASET_DIR = Path("Datasets/RQD_remapped").resolve()

MODEL_WEIGHTS = "src/yolo11x-cls.pt"

EPOCHS = 50

PATIENCE = 10

IMGSZ = 224
BATCH = 32
DEVICE = 0  # or "cpu"

PROJECT = Path.cwd() / "ModelTraining/RQD-remapped_YOLO11x"

PHASE_NAME = "RQD-remapped_train"
TEST_NAME   = "RQD-remapped_test"

RANDOM_SEED = 42

In [ ]:
common_args = {
    "data": str(DATASET_DIR),
    "imgsz": IMGSZ,
    "batch": BATCH,
    "device": DEVICE,

    "patience": PATIENCE,

    "seed": RANDOM_SEED,
    "deterministic": True,

    "plots": True,
    "verbose": True,

    "project": PROJECT,
    "exist_ok": True,
}

In [ ]:
print("\n" + "=" * 70)
print("PHASE - FINE-TUNE COMPLETE YOLO11")
print("=" * 70)

model = YOLO(MODEL_WEIGHTS)


results_phase = model.train(
    **common_args,
    epochs=EPOCHS,
    name=PHASE_NAME,

    # No layers frozen.
    freeze=None
)


phase_dir = Path(model.trainer.save_dir)

phase_best = phase_dir / "weights" / "best.pt"


print("\nPhase directory:")
print(phase_dir)

print("\nBest model:")
print(phase_best)


if not phase_best.exists():
    raise FileNotFoundError(
        f"Best model was not found: {phase_best}"
    )

In [ ]:
print("\n" + "=" * 70)
print("FINAL TEST SET EVALUATION")
print("=" * 70)


best_model = YOLO(str(phase_best))


test_metrics = best_model.val(
    data=str(DATASET_DIR),

    split="test",

    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,

    project=PROJECT,
    name=TEST_NAME,

    plots=True
)


print("\n" + "=" * 70)
print("FINAL YOLO11 TEST RESULTS")
print("=" * 70)

print(
    f"Top-1 Test Accuracy: "
    f"{test_metrics.top1:.4f} "
    f"({test_metrics.top1 * 100:.2f}%)"
)

print(
    f"Top-5 Test Accuracy: "
    f"{test_metrics.top5:.4f} "
    f"({test_metrics.top5 * 100:.2f}%)"
)

In [ ]:
phase_csv = phase_dir / "results.csv"


df = pd.read_csv(phase_csv)

df.columns = df.columns.str.strip()


print("\nPhase columns:")
print(df.columns.tolist())

In [ ]:
val_acc = (
    df["metrics/accuracy_top1"].tolist()
)

train_loss = (
    df["train/loss"].tolist()
)

val_loss = (
    df["val/loss"].tolist()
)


epochs = range(1, len(val_acc) + 1)


In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    val_acc,
    label="Validation Top-1 Accuracy"
)


plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title("YOLO11x Validation Accuracy - Remapped")

plt.legend()

plt.xlim(left=0)

plt.tight_layout()

plt.show()


In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    train_loss,
    label="Train Loss"
)

plt.plot(
    epochs,
    val_loss,
    label="Validation Loss"
)


plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("YOLO11x Loss - Remapped")

plt.xlim(left=0)

plt.legend()

plt.tight_layout()

plt.show()